# Do compressed code LLMs stay consistent?

Runs the full experiment on a free Colab GPU: Qwen2.5-Coder at FP16, 8-bit and 4-bit on HumanEval,
with four meaning-preserving prompt changes.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Pan-afs/compressed-llm-consistency.git
%cd compressed-llm-consistency
!pip install -q -r requirements.txt

### Where to keep raw results

Colab sessions can disconnect. The run script appends results after every batch and resumes
where it stopped, so it helps to keep the raw files on Google Drive. Set `USE_DRIVE = False`
to keep them in the session instead.

In [ ]:
USE_DRIVE = True
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # or "Qwen/Qwen2.5-Coder-0.5B-Instruct" for a faster run

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW = "/content/drive/MyDrive/compressed-llm-consistency/raw"
else:
    RAW = "results/raw"
print(RAW)

### 1. Check the perturbations

Every perturbation is applied to `prompt + reference solution`; all reference solutions must still pass.

In [ ]:
!python -m codecons.check_perturbations

### 2. Smoke test (a few problems, FP16 only)

In [ ]:
!python -m codecons.run --model {MODEL} --precision fp16 --limit 3 --out /tmp/smoke

### 3. Full run

One cell per precision so each model is loaded in a fresh process. On a T4 the 8-bit run is the slowest
(LLM.int8() kernels are not optimised for speed). Re-run a cell after a disconnect to resume.

In [ ]:
!python -m codecons.run --model {MODEL} --precision fp16 --out {RAW}

In [ ]:
!python -m codecons.run --model {MODEL} --precision int8 --out {RAW}

In [ ]:
!python -m codecons.run --model {MODEL} --precision int4 --out {RAW}

### 4. Tables and chart

In [ ]:
!python -m codecons.analyze --raw {RAW} --update-readme

In [ ]:
from IPython.display import Image, Markdown, display

display(Image("results/consistency.png"))
display(Markdown(open("results/results.md").read()))

### 5. Download the results

Commit `results/` and the updated `README.md` to the repository.

In [ ]:
!zip -qr results.zip results README.md -x "results/raw/*"
from google.colab import files
files.download("results.zip")